## Gold DDL
Creates the gold schema and all gold tables (empty) with data types,
comments and primary/foreign key constraints.
Safe to re-run: uses `IF NOT EXISTS`, so existing data is never dropped.
Data is loaded separately by `02_gold_dimensions`, `03_gold_facts`, `04_gold_aggregates`.

In [0]:
%run ./00_config

### dim_category

In [0]:
spark.sql(f"""
         CREATE TABLE IF NOT EXISTS {gold['dim_category']} (
             category_id INT NOT NULL           COMMENT 'YouTube category code from the source CSV (e.g. 10)',
             category_name STRING NOT NULL      COMMENT 'Human-readable category name (e.g. Music)',
             _updated_at TIMESTAMP              COMMENT 'When this row was last inserted or changed by the gold load',
             CONSTRAINT pk_dim_category PRIMARY KEY (category_id) 
         )
         COMMENT 'Static reference dimension: YouTube category code -> name (US region). One row per category.'
""")

### dim_channel

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold['dim_channel']} (
        channel_key BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY   COMMENT 'Surrogate key generated by Delta (1, 2, 3...)',
        channel_title STRING NOT NULL                              COMMENT 'YouTube channel name (business key), e.g. Saturday Night Live',
        _updated_at TIMESTAMP                                      COMMENT 'When this row was inserted by the gold load',
        CONSTRAINT pk_dim_channel PRIMARY KEY (channel_key)
    )
    COMMENT 'Channel dimension. One row per YouTube channel that appeared in trending.'
""")

### dim_video

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold['dim_video']} (
        video_id STRING NOT NULL              COMMENT 'YouTube video ID (natural key, never changes), e.g. 2kyS6SvSYSE',
        title STRING                          COMMENT 'Latest known video title',
        publish_ts TIMESTAMP                  COMMENT 'When the video was published on YouTube (UTC)',
        tags STRING                           COMMENT 'Latest known tags, pipe-separated as in the source',
        comments_disabled BOOLEAN             COMMENT 'True if comments are disabled for the video',
        ratings_disabled BOOLEAN              COMMENT 'True if likes/dislikes are hidden for the video',
        _updated_at TIMESTAMP                  COMMENT 'When this row was last inserted or changed by the gold load',
        CONSTRAINT pk_dim_video PRIMARY KEY (video_id)
    )
    COMMENT 'Video dimension (SCD Type 1: latest attributes win). One row per video_id.'
""")

### dim_date

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold['dim_date']} (
        date_key INT NOT NULL            COMMENT 'Date key in yyyyMMdd format, e.g. 20171114',
        full_date DATE NOT NULL          COMMENT 'Calendar date',
        year INT                         COMMENT 'Year, e.g. 2017',
        quarter INT                      COMMENT 'Quarter of the year: 1-4',
        month INT                        COMMENT 'Month number: 1-12',
        month_name STRING                COMMENT 'Month name, e.g. November',
        day_of_month INT                 COMMENT 'Day of the month: 1-31',
        day_of_week INT                  COMMENT 'Day of the week: 1 = Sunday ... 7 = Saturday (Spark convention)',
        day_name STRING                  COMMENT 'Day name, e.g. Tuesday',
        is_weekend BOOLEAN               COMMENT 'True for Saturday and Sunday',
        _updated_at TIMESTAMP            COMMENT 'When this row was inserted by the gold load',
        CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
    )
    COMMENT 'Calendar dimension. One row per day covering the trending date range.'
""")

### fact_video_daily_stats

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold['fact_video_daily_stats']} (
        date_key INT NOT NULL               COMMENT 'Trending day (FK to dim_date), yyyyMMdd',
        video_id STRING NOT NULL            COMMENT 'Video (FK to dim_video)',
        channel_key BIGINT NOT NULL         COMMENT 'Channel (FK to dim_channel)',
        category_id INT NOT NULL            COMMENT 'Category (FK to dim_category)',
        views BIGINT                        COMMENT 'Cumulative views on this trending day. Semi-additive: do NOT sum across days for the same video',
        likes BIGINT                        COMMENT 'Cumulative likes on this trending day (semi-additive)',
        dislikes BIGINT                     COMMENT 'Cumulative dislikes on this trending day (semi-additive)',
        comment_count BIGINT                COMMENT 'Cumulative comments on this trending day (semi-additive)',
        views_delta BIGINT                  COMMENT 'Views gained since the previous trending day of the same video. NULL on the first trending day. Additive: safe to SUM',
        trending_day_num INT                COMMENT 'Sequential number of this day in the video trending history: 1 = first day in trending',
        _updated_at TIMESTAMP               COMMENT 'When this row was last inserted or changed by the gold load',
        CONSTRAINT pk_fact_video_daily_stats PRIMARY KEY (video_id, date_key),
        CONSTRAINT fk_fact_date FOREIGN KEY (date_key) REFERENCES {gold['dim_date']}(date_key),
        CONSTRAINT fk_fact_video FOREIGN KEY (video_id) REFERENCES {gold['dim_video']}(video_id),
        CONSTRAINT fk_fact_channel FOREIGN KEY (channel_key) REFERENCES {gold['dim_channel']}(channel_key),
        CONSTRAINT fk_fact_category FOREIGN KEY (category_id) REFERENCES {gold['dim_category']}(category_id)
    )
    COMMENT 'Periodic snapshot fact. Grain: one row per video per trending day.'
""")

### agg_category_daily

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold['agg_category_daily']} (
        date_key            INT       NOT NULL COMMENT 'Day key, yyyyMMdd',
        full_date           DATE      NOT NULL COMMENT 'Calendar date',
        category_id         INT       NOT NULL COMMENT 'YouTube category code',
        category_name       STRING             COMMENT 'Category name, e.g. Music',
        videos_in_trending  BIGINT             COMMENT 'How many videos of this category were trending on this day',
        views_gained        BIGINT             COMMENT 'Views gained on this day (sum of views_delta)',
        like_rate           DOUBLE             COMMENT 'Share of views that ended with a like: SUM(likes) / SUM(views)',
        _updated_at         TIMESTAMP          COMMENT 'When this row was last rebuilt',
        CONSTRAINT pk_agg_category_daily PRIMARY KEY (category_id, date_key)
    )
    COMMENT 'Daily KPIs per category. One row per category per day.'
""")

### agg_channel_summary

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold['agg_channel_summary']} (
        channel_key         BIGINT    NOT NULL COMMENT 'Channel surrogate key',
        channel_title       STRING             COMMENT 'Channel name',
        videos_trended      BIGINT             COMMENT 'How many videos of the channel appeared in trending',
        avg_days_per_video  DOUBLE             COMMENT 'Average number of days a video of the channel stayed in trending',
        total_views         BIGINT             COMMENT 'Sum of each video peak (max) views — correct total for cumulative views',
        _updated_at         TIMESTAMP          COMMENT 'When this row was last rebuilt',
        CONSTRAINT pk_agg_channel_summary PRIMARY KEY (channel_key)
    )
    COMMENT 'Channel KPIs over the whole period. One row per channel.'
""")